# PRUNING

In [2]:
!pip install -q torch torchvision wandb

In [3]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')


import wandb
wandb.login(WANDB_API_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sagar-sharma-03015611924 (practicum-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
!git clone https://github.com/bremsstrahlung-57/practicum-project/

Cloning into 'practicum-project'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 29 (delta 7), reused 23 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 39.70 MiB | 17.49 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [5]:
import torch
import numpy as np
import torchvision
import time
import tracemalloc
import os
import copy

In [10]:
def get_prunable_params(model):
    return [
        (m, "weight")
        for m in model.modules()
        if isinstance(m, (torch.nn.Conv2d, torch.nn.Linear))
    ]

def actual_sparsity(model):
    zeros = total = 0
    for m, _ in get_prunable_params(model):
        w = m.weight.data
        zeros += (w == 0).sum().item()
        total += w.numel()
    return zeros / total

def eval_accuracy_cpu(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total

def benchmark_cpu(model, n_warmup=20, n_runs=100):
    model.eval()
    dummy = torch.randn(1,3,32,32)

    # warmup
    for _ in range(n_warmup):
        with torch.no_grad():
            model(dummy)

    # latency
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        with torch.no_grad():
            model(dummy)
        times.append((time.perf_counter() - t0) * 1000)

    # ram
    tracemalloc.start()
    with torch.no_grad():
        model(dummy)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return np.mean(times), np.std(times), peak / 1024**2

def save_model_and_size(model, path):
    torch.save(model.state_dict(), path)
    return os.path.getsize(path) / 1024**2 #MB

In [7]:
import torchvision.transforms as transforms

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)),
])
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256,shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:08<00:00, 20.4MB/s]


In [8]:
def get_cifar_resnet18():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = torch.nn.Identity()
    model.fc = torch.nn.Linear(512, 10)
    return model

CKPT_PATH = "/content/practicum-project/resnet18_cifar10_baseline.pth"
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_baseline():
    m = get_cifar_resnet18()
    m.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    return m


In [11]:
import torch.nn.utils.prune as prune

SPARSITY_LEVELS = [0.10, 0.30, 0.50, 0.70, 0.90]

wandb.init(project="cnn-compression", name="pruning-sweep", id="pruning-sweep",resume="allow")

for target_sparsity in SPARSITY_LEVELS:
    print(f"\n{'='*50}")
    print(f"  Sparsity target: {target_sparsity*100:.0f}%")
    print(f"{'='*50}")

    # fresh copy of baseline each time — one-shot pruning per level
    model = load_baseline().to(DEVICE)
    params_to_prune = get_prunable_params(model)

    # apply global magnitude pruning
    prune.global_unstructured(
        params_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=target_sparsity,
    )

    # make permanent (removes forward hooks + mask buffers, zeros stay)
    for module, _ in params_to_prune:
        prune.remove(module, "weight")

    measured_sparsity = actual_sparsity(model)
    print(f"  Measured sparsity: {measured_sparsity*100:.2f}%")

    # move to CPU for all eval/benchmarking
    model_cpu = model.cpu()

    acc = eval_accuracy_cpu(model_cpu, testloader)
    lat_mean, lat_std, ram_mb = benchmark_cpu(model_cpu)
    size_mb = save_model_and_size(model_cpu,f"pruned_{int(target_sparsity*100)}.pth")

    print(f"  Accuracy:  {acc:.2f}%")
    print(f"  Latency:   {lat_mean:.2f} ± {lat_std:.2f} ms")
    print(f"  RAM:       {ram_mb:.2f} MB")
    print(f"  Size:      {size_mb:.2f} MB")

    wandb.log({
        "sparsity":          measured_sparsity,
        "accuracy":          acc,
        "latency_mean_ms":   lat_mean,
        "latency_std_ms":    lat_std,
        "ram_mb":            ram_mb,
        "model_size_mb":     size_mb,
    })

wandb.finish()



  Sparsity target: 10%
  Measured sparsity: 10.00%
  Accuracy:  95.23%
  Latency:   20.51 ± 2.05 ms
  RAM:       0.00 MB
  Size:      42.70 MB

  Sparsity target: 30%
  Measured sparsity: 30.00%
  Accuracy:  95.27%
  Latency:   23.64 ± 5.34 ms
  RAM:       0.00 MB
  Size:      42.70 MB

  Sparsity target: 50%
  Measured sparsity: 50.00%
  Accuracy:  95.18%
  Latency:   20.50 ± 3.30 ms
  RAM:       0.00 MB
  Size:      42.70 MB

  Sparsity target: 70%
  Measured sparsity: 70.00%
  Accuracy:  94.83%
  Latency:   19.34 ± 1.43 ms
  RAM:       0.00 MB
  Size:      42.70 MB

  Sparsity target: 90%
  Measured sparsity: 90.00%
  Accuracy:  42.00%
  Latency:   27.16 ± 4.79 ms
  RAM:       0.00 MB
  Size:      42.70 MB


accuracy,████▁
latency_mean_ms,▂▅▂▁█
latency_std_ms,▂█▄▁▇
model_size_mb,▁▁▁▁▁
ram_mb,▁▁▁▁▁
sparsity,▁▃▄▆█
accuracy,42
latency_mean_ms,27.16461
latency_std_ms,4.79416
model_size_mb,42.69941
ram_mb,0.0023
